In [7]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_absolute_error, r2_score

from xgboost import XGBClassifier, XGBRegressor

# Load data
df = pd.read_csv("QCFE2_Perfect_Inventory.csv")

df['Order_date'] = pd.to_datetime(df['Order_date'])

In [8]:
categorical_cols = [
    'City', 'Company', 'Product_Category',
    'Payment_Method', 'Season', 'Day_of_Week'
]

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [9]:
features = [
    'City',
    'Product_Category',
    'Season',
    'Is_Festival',
    'Day_of_Week',
    'Is_Weekend',
    'Month',
    'Hour',
    'Peak_Intensity',
    'Distance_Km',
    'Delivery_Time_Minute',
    'Supplier_Lead_Time'
]

df = df.dropna()

# Convert any object columns
for col in features:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('category').cat.codes

In [10]:
# ==========================================
# STOCK-OUT DECISION ENGINE (NO ML, NO LEAKAGE)
# ==========================================

# This function uses:
# - predicted_demand (from your demand forecasting model)
# - current_stock (real-time input)
# - safety_stock (optional buffer)

def inventory_decision(predicted_demand, current_stock, safety_stock=0):

    # -----------------------------
    # Total required stock
    # -----------------------------
    required_stock = predicted_demand + safety_stock

    # -----------------------------
    # Status logic
    # -----------------------------
    if current_stock < predicted_demand:
        status = "Stock-Out Risk"

    elif current_stock > predicted_demand * 1.5:
        status = "Overstock"

    else:
        status = "Sufficient"

    # -----------------------------
    # Risk calculation (%)
    # -----------------------------
    if predicted_demand == 0:
        risk = 0
    else:
        risk = max(0, (predicted_demand - current_stock) / predicted_demand * 100)

    # -----------------------------
    # Suggested action
    # -----------------------------
    reorder_qty = max(0, required_stock - current_stock)

    return {
        "status": status,
        "risk_percentage": round(risk, 2),
        "required_stock": int(required_stock),
        "reorder_quantity": int(reorder_qty)
    }


# ==========================================
# SAMPLE TEST (REMOVE LATER IN STREAMLIT)
# ==========================================

predicted_demand = 400   # from your demand model
current_stock = 250      # input from user / dataset
safety_stock = 50

result = inventory_decision(predicted_demand, current_stock, safety_stock)

print("\n===== INVENTORY DECISION =====")
for k, v in result.items():
    print(f"{k}: {v}")


===== INVENTORY DECISION =====
status: Stock-Out Risk
risk_percentage: 37.5
required_stock: 450
reorder_quantity: 200


In [11]:
# ==========================================
# OPTIMAL STOCK MODEL (IMPROVED)
# ==========================================

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# -----------------------------
# CREATE TARGET
# -----------------------------
df['Optimal_Stock'] = df['Items_Count'] * df['Supplier_Lead_Time']

# -----------------------------
# FEATURE ENGINEERING (KEY FIX)
# -----------------------------
df['Demand_LeadTime'] = df['Items_Count'] * df['Supplier_Lead_Time']
df['Demand_per_Hour'] = df['Items_Count'] / (df['Hour'] + 1)
df['Demand_Intensity'] = df['Items_Count'] * df['Peak_Intensity']

# -----------------------------
# UPDATED FEATURES
# -----------------------------
features = [
    'City',
    'Product_Category',
    'Season',
    'Is_Festival',
    'Day_of_Week',
    'Is_Weekend',
    'Month',
    'Hour',
    'Peak_Intensity',
    'Distance_Km',
    'Delivery_Time_Minute',
    'Supplier_Lead_Time',

    # ✅ IMPORTANT SIGNALS
    'Items_Count',
    'Demand_LeadTime',
    'Demand_per_Hour',
    'Demand_Intensity'
]

# Clean data
df = df.dropna()

# Convert objects
for col in features:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('category').cat.codes

# -----------------------------
# SPLIT
# -----------------------------
X = df[features]
y = df['Optimal_Stock']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# MODEL (TUNED)
# -----------------------------
model_optimal = XGBRegressor(
    n_estimators=600,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Train
model_optimal.fit(X_train, y_train)

# Predict
y_pred = model_optimal.predict(X_test)

# -----------------------------
# EVALUATION
# -----------------------------
print("\n===== OPTIMAL STOCK MODEL =====")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))


===== OPTIMAL STOCK MODEL =====
MAE: 0.012483437545597553
R2 Score: 0.9999982118606567


In [12]:
# joblib.dump(model_stockout, "stockout_model.pkl")
# joblib.dump(model_optimal, "optimal_stock_model.pkl")
# joblib.dump(label_encoders, "label_encoders.pkl")

# print("✅ Models saved!")